<a href="https://colab.research.google.com/github/salty-arch/Flyrank-Intern/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule**: 'A page is worth reviewing for a title rewrite if the gap pages CTR is below the average CTR of its position tier of that page and if the page is getting enough impressions but not enough CTR on par to those impressions.'

**Reason Code**: low_ctr_with_high_impressions (one reason code because the rule noly encodes one diagnosis in which CTR and Impressions both are required).

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
query = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

query["ctr"] = query["total_clicks"] / query["total_impressions"]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
def assign_tier(pos):
    if pos <= 3:
        return "top_3"
    elif pos <= 10:
        return "page_1"
    elif pos <= 20:
        return "striking"
    elif pos <= 50:
        return "page_3_5"
    else:
        return "deep"

query["position_tier"] = query["avg_position"].apply(assign_tier)

In [12]:
tier_medians = query[query["ctr"] > 0].groupby("position_tier")["ctr"].median()
print(tier_medians)

position_tier
deep        0.005634
page_1      0.003398
page_3_5    0.002032
striking    0.003178
top_3       0.003507
Name: ctr, dtype: float64


In [13]:

print(query["position_tier"].value_counts())

position_tier
page_1      81988
page_3_5    33288
striking    32203
top_3       17578
deep        11681
Name: count, dtype: int64


In [14]:
query["tier_median_ctr"] = query["position_tier"].map(tier_medians)
query["gap"] = query["tier_median_ctr"] - query["ctr"]
query["priority_score"] = query["gap"] * query["total_impressions"]
query["reason_code"] = "low_ctr_with_high_impressions"
query["action"] = "rewrite_title"

top20 = query.sort_values("priority_score", ascending=False).head(20)
top20[["content_hash_id", "client_hash_id", "position_tier", "ctr", "tier_median_ctr", "gap", "total_impressions", "priority_score", "reason_code", "action"]]

,content_hash_id,client_hash_id,position_tier,ctr,tier_median_ctr,gap,total_impressions,priority_score,reason_code,action
96420,content_44f34c0a90047651,client_23a62021009f63c4,page_1,0.000113,0.003398,0.003285,212404.0,697.848768,low_ctr_with_high_impressions,rewrite_title
54365,content_8e1334d6356668e3,client_73cda7b4e4f265ea,page_1,0.000007,0.003398,0.003391,134984.0,457.739167,low_ctr_with_high_impressions,rewrite_title
136658,content_34a70fea29d15f24,client_62f4a7e64f5e0096,page_1,0.000301,0.003398,0.003098,143019.0,443.045879,low_ctr_with_high_impressions,rewrite_title
24110,content_8d7d99f109e19aa2,client_e547b89c05043229,top_3,0.001420,0.003507,0.002087,203497.0,424.633529,low_ctr_with_high_impressions,rewrite_title
142934,content_fec55986a1868d62,client_73cda7b4e4f265ea,page_1,0.000008,0.003398,0.003390,124075.0,420.665251,low_ctr_with_high_impressions,rewrite_title
130208,content_7c6373141eae744a,client_62f4a7e64f5e0096,page_1,0.000626,0.003398,0.002772,132593.0,367.613424,low_ctr_with_high_impressions,rewrite_title
101102,content_f6116743b00afc2d,client_62f4a7e64f5e0096,page_1,0.000139,0.003398,0.003259,107584.0,350.621071,low_ctr_with_high_impressions,rewrite_title
130236,content_acbcc847f8996314,client_62f4a7e64f5e0096,page_1,0.001534,0.003398,0.001865,170808.0,318.485981,low_ctr_with_high_impressions,rewrite_title
89120,content_cd3d932d4e1c8db0,client_9958f0a7ae1df715,page_1,0.000045,0.003398,0.003354,89332.0,299.592184,low_ctr_with_high_impressions,rewrite_title
130406,content_b99ea6861864dea5,client_62f4a7e64f5e0096,page_1,0.001858,0.003398,0.001541,194337.0,299.448598,low_ctr_with_high_impressions,rewrite_title


In [15]:
import os
os.makedirs("/content/FlyRank-Intern/work/outputs", exist_ok=True)

output_cols = ["content_hash_id", "client_hash_id", "position_tier", "ctr",
               "tier_median_ctr", "gap", "total_impressions", "priority_score",
               "reason_code", "action"]

ranked_queue = query.sort_values("priority_score", ascending=False)[output_cols]
ranked_queue.to_csv("/content/FlyRank-Intern/work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(ranked_queue)} rows to /content/FlyRank-Intern/work/outputs/baseline_action_score.csv")

Wrote 176738 rows to /content/FlyRank-Intern/work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



For every row: **action**, **reason code**, **confidence**, and **what would make it wrong**.

Action and reason code are the same for every row by design, my rule only ever
encodes one diagnosis (see Section 1), so every flagged page gets the same
recommendation. What varies is the strength of the evidence (confidence) and the
specific risk that this particular pick could be a false alarm.

**Confidence rule used:** large gap (≥0.0030) + impressions ≥100K → High.
Large gap + impressions <100K → Medium-High. Smaller gaps get Medium/Low
regardless of impressions, since the gap itself is the weaker signal at that point.

---

**1. `content_44f34c0a90047651`** (page_1): CTR 0.011% vs tier median 0.34%, impressions 212,404
Action: rewrite_title | Reason: low_ctr_with_high_impressions | **Confidence: High**
What would make it wrong: could reflect a tracking/measurement issue rather than real user behavior.

**2. `content_8e1334d6356668e3`** (page_1): CTR 0.0007% vs 0.34%, impressions 134,984
**Confidence: High**
What would make it wrong: searchers may get their answer from the snippet without clicking through, title isn't the issue.

**3. `content_34a70fea29d15f24`** (page_1): CTR 0.03% vs 0.34%, impressions 143,019
**Confidence: Medium**
What would make it wrong: page may have been recently updated/republished, CTR not yet stabilized.

**4. `content_8d7d99f109e19aa2`** (top_3): CTR 0.14% vs 0.35%, impressions 203,497
**Confidence: Medium**
What would make it wrong: page already holds a valuable top_3 position, rewriting risks disrupting that ranking for uncertain CTR gain.

**5. `content_fec55986a1868d62`** (page_1): CTR 0.0008% vs 0.34%, impressions 124,075
**Confidence: High**
What would make it wrong: possible topic/intent mismatch, page may be ranking for the wrong query entirely, so no title fix would help.

**6. `content_7c6373141eae744a`** (page_1): CTR 0.0626% vs 0.34%, impressions 132,593
**Confidence: Medium**
What would make it wrong: could reflect a tracking/measurement issue rather than real user behavior.

**7. `content_f6116743b00afc2d`** (page_1): CTR 0.0139% vs 0.34%, impressions 107,584
**Confidence: Medium**
What would make it wrong: searchers may be satisfied by the snippet without clicking through.

**8. `content_acbcc847f8996314`** (page_1): CTR 0.15% vs 0.34%, impressions 170,808
**Confidence: Low**
What would make it wrong: with a gap this size, CTR may fall within normal variation for the tier, not every below-median page is a real outlier.

**9. `content_cd3d932d4e1c8db0`** (page_1): CTR 0.0045% vs 0.34%, impressions 89,332
**Confidence: Medium-High**
What would make it wrong: could reflect a tracking/measurement issue rather than real user behavior.

**10. `content_b99ea6861864dea5`** (page_1): CTR 0.19% vs 0.34%, impressions 194,337
**Confidence: Medium-Low**
What would make it wrong: with a gap this size, CTR may fall within normal variation for the tier.

**11. `content_f43118e089ecc69a`** (page_1): CTR 0.137% vs 0.34%, impressions 139,417
**Confidence: Low**
What would make it wrong: with a gap this size, CTR may fall within normal variation for the tier.

**12. `content_046fc480045b88f5`** (page_1): CTR 0.0072% vs 0.34%, impressions 83,788
**Confidence: Medium-High**
What would make it wrong: extremely low CTR despite confirmed visibility could point to a tracking/measurement issue rather than a genuine title problem.

**13. `content_9540d884af3e41fd`** (page_1): CTR 0.0134% vs 0.34%, impressions 82,376
**Confidence: Medium-High**
What would make it wrong: could reflect a tracking/measurement issue rather than real user behavior.

**14. `content_9c057b66c30a3abb`** (striking): CTR 0.0012% vs tier median 0.32%, impressions 83,834
**Confidence: Medium-High**
What would make it wrong: the `striking` tier's own median CTR showed unusual, inconsistent patterns in earlier (Week 1) analysis, higher than `top_3`'s median. Comparing this page against a possibly unreliable tier baseline is less trustworthy than for other tiers.

**15. `content_306bc78dff1eb683`** (top_3): CTR 0.043% vs 0.35%, impressions 80,821
**Confidence: High**
What would make it wrong: page already holds a valuable top_3 position, rewriting risks disrupting that ranking.

**16. `content_425715547c6a3ea8`** (page_1): CTR 0.0042% vs 0.34%, impressions 71,513
**Confidence: Medium-High**
What would make it wrong: page may have been recently updated/republished, CTR not yet stabilized.

**17. `content_e578ac84778da489`** (page_1): CTR 0.138% vs 0.34%, impressions 117,764
**Confidence: Low**
What would make it wrong: with a gap this size, CTR may fall within normal variation for the tier.

**18. `content_36fc1ee501ec072d`** (page_1): CTR 0.022% vs 0.34%, impressions 73,135
**Confidence: Medium-High**
What would make it wrong: page may have been recently updated/republished, CTR not yet stabilized.

**19. `content_82e35c4845e6c391`** (page_3_5): CTR 0.042% vs tier median 0.20%, impressions 143,907
**Confidence: Medium**
What would make it wrong: `page_3_5` already has a much lower typical CTR than higher tiers, pages ranked further down naturally get fewer clicks. This page may simply be performing normally for a lower-visibility tier.

**20. `content_4977e90c4d93cf9f`** (page_1): CTR 0.038% vs 0.34%, impressions 73,862
**Confidence: Medium-High**
What would make it wrong: searchers may be satisfied by the snippet without clicking through.

---

**Tier distribution in the top 20:** 16 page_1, 2 top_3, 1 striking, 1 page_3_5.
This is a real pattern worth naming explicitly in Section 4, my formula
(gap × impressions) structurally favors tiers with higher realistic impression
ceilings, meaning genuinely bad titles on lower-visibility tiers (`deep`,
`striking`) may be under-flagged even if their CTR gap is just as large in
relative terms.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Which picks look wrong, and why:**

The weakest picks in my top 20 are rows 8, 11, and 17 (Low confidence)
`content_acbcc847f8996314`, `content_f43118e089ecc69a`, and `content_e578ac84778da489`.
All three have noticeably smaller CTR gaps than the rest of the list
(~0.0018-0.0020 vs. 0.003+ for most other rows). With a gap this small, the
page's CTR may simply fall within normal variation for its tier rather than
representing a genuine title problem, these are the picks I'd trust least
if I only had time to review a handful.

Row 14 (`content_9c057b66c30a3abb`, striking tier) is also worth flagging
as a weak pick for a different reason: the `striking` tier's own median CTR
showed unusual, inconsistent behavior in my Week 1 EDA (higher than `top_3`'s
median). If the tier baseline itself is unreliable, comparing this page
against it is shakier than for the other tiers.

More broadly, my top 20 is heavily skewed toward `page_1` (16 of 20 rows),
with only 2 `top_3`, 1 `striking`, and 1 `page_3_5`. This isn't because
`page_1` pages have uniquely bad titles — it's a structural artifact of my
formula (gap × impressions): lower-visibility tiers like `deep` and `striking`
have lower impression ceilings, so even a page with a large CTR gap in that
tier can't score as high as a `page_1` page with the same gap and more
impressions. Genuinely bad titles in lower tiers are likely under-flagged
by this rule.

**Leakage check:**

My score uses only `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (all
summed/averaged over March), and values derived from them (`ctr`,
`position_tier`, `tier_median_ctr`, `gap`, `priority_score`). None of these
require knowledge of anything after the scoring moment — they're all
observed performance during the window being scored, not future outcomes.

I did not use any product/access flags (e.g. `client_has_gsc`, `gsc_data_available`,
`client_has_ga4`, `ga4_data_available`) as inputs to the model, only as filters to
remove rows with insufficient data (Week 3, Section 2). I did not use any engagement
metrics from GA4, channel-wise session breakdowns, or AI referral columns in my
analysis - all of these were explicitly stated as irrelevant to this line of research
in Week 3, and none of these columns were used anywhere in this week's score
calculation. No metrics from future dates (April, May, June) were used to calculate
any of the metrics in March.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.